### 랭체인 준비

In [ ]:
# 패키지 설치
!pip install langchain==0.3.7
!pip install langchain-google-genai
!pip install langchain_community
!pip install langgraph

In [ ]:
import os
from google.colab import userdata

# 환경 변수 준비(좌측 상단의 열쇠 아이콘으로 GOOGLE_API_KEY 설정)
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

###랭스미스

In [ ]:
import os
from uuid import uuid4

# 환경 변수 준비
unique_id = uuid4().hex[0:8]
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = f"Tracing Walkthrough - {unique_id}"

### 임베딩 모델 준비

In [ ]:
# 허깅 페이스 패키지 설치
!pip install langchain-huggingface

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# 임베딩 모델 준비
hf_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
)

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### 도구 준비

#### 타빌리 도구 준비

In [ ]:
import os
from google.colab import userdata

# 환경 변수 준비(좌측 상단의 열쇠 아이콘으로 TAVILY_API_KEY 설정)
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# 타빌리 도구 준비
tavily_tool = TavilySearchResults(max_results=2)

In [ ]:
# 타빌리 도구 실행
tavily_tool.invoke("현재 서울의 날씨는?")

[{'url': 'https://www.accuweather.com/ko/kr/seoul/1-226081_30_al/current-weather/1-226081_30_al',
  'content': '서울, 서울시, 대한민국 지역의 레이더 예보와 시간별 예보, 현재 날씨 예보로 기상 상황에 대비하실 수 있습니다. 날씨에 따라 하루를 계획하세요'},
 {'url': 'https://weather.com/ko-KR/weather/today/l/82e46175f97c224acf6b95afc4934fbae0e4ba123adcee8a52b7be97c303467b',
  'content': 'The Weather Channel 및 Weather.com이 제공하는 오늘과 오늘 밤 서울특별시 일기예보, 날씨 상태 및 도플러 레이더'}]

#### 검색 도구 준비

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 준비
loader = WebBaseLoader("https://python.langchain.com/")
documents = loader.load()

In [ ]:
print(len(documents))

1


In [ ]:
# 문서 분할
documents = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
).split_documents(documents)
print(len(documents))

17


In [ ]:
# 랭체인 크로마 패키지 설치
!pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

# 벡터 스토어 준비
vectorstore = Chroma.from_documents(
    documents,
    embedding=hf_embeddings,
)

# 검색 도구 준비
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

In [ ]:
from langchain.tools.retriever import create_retriever_tool

# 검색 도구 준비
retriever_tool = create_retriever_tool(
    retriever,
    "langchain_search",
    "랭체인에 관한 정보를 검색합니다. 랭체인에 관해 질문이 있다면, 이 도구를 사용해주세요.",
)

In [ ]:
# 검색 도구 실행
retriever_tool.invoke("랭스미스에 관해 알려주세요.")

'(self-constructing) chainText embedding modelsHow to combine results from multiple retrieversHow to select examples from a LangSmith datasetHow to select examples by lengthHow to select examples by maximal marginal relevance (MMR)How to select examples by n-gram overlapHow to select examples by similarityHow to use reference examples when doing extractionHow to handle long text when doing extractionHow to use prompting alone (no tool calling) to do extractionHow to add fallbacks to a runnableHow to filter messagesHybrid SearchHow to use the LangChain indexing APIHow to inspect runnablesLangChain Expression Language CheatsheetHow to cache LLM responsesHow to track token usage for LLMsRun models locallyHow to get log probabilitiesHow to reorder retrieved results to mitigate the "lost in the middle" effectHow to split Markdown by HeadersHow to merge consecutive messages of the same typeHow to add message historyHow to migrate from legacy LangChain agents to LangGraphHow to retrieve using

### 에이전트 구현

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 준비
llm = ChatGoogleGenerativeAI(
    model="models/gemini-1.5-flash",
)

In [ ]:
from langgraph.prebuilt import chat_agent_executor

# 에이전트 준비
agent_executor = chat_agent_executor.create_tool_calling_executor(
    llm,
    tools=[tavily_tool, retriever_tool]
)

In [ ]:
# 도구를 사용하지 않은 질의 응답
response = agent_executor.invoke(
    {"messages": [("human", "안녕하세요.")]}
)
response["messages"][-1].content

'안녕하세요! 무엇을 도와드릴까요? 😊 \n'

In [ ]:
# 타빌리 도구를 사용한 질의 응답
response = agent_executor.invoke(
    {"messages": [("human", "현재 서울의 날씨는?")]}
)
response["messages"][-1].content

"죄송합니다. 현재 날씨 정보를 제공할 수 없습니다.  다만, 서울의 날씨를 알아보기 위해 'freemeteo.kr' 또는 'accuweather.com'과 같은 웹사이트를 방문하는 것을 추천합니다. \n"

In [ ]:
# 검색 도구를 사용한 질의 응답
response = agent_executor.invoke(
    {"messages": [("human", "랭스미스에 관해 알려주세요.")]}
)
response["messages"][-1].content

'랭스미스는 랭체인의 데이터셋 관리 및 실험을 위한 도구입니다. 랭스미스를 사용하면 랭체인 모델을 훈련하고 평가하는 데 필요한 데이터를 관리하고, 실험 결과를 추적하고 분석할 수 있습니다. 랭스미스는 랭체인의 기능을 확장하고, 랭체인 모델을 더 효과적으로 개발하는 데 도움이 됩니다.'

### 메시지 스트리밍

In [ ]:
# 메시지 스트리밍
for chunk in agent_executor.stream(
    {"messages": [("human", "랭스미스에 관해 알려주세요.")]}
):
    print(chunk)
    print("--")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'langchain_search', 'arguments': '{"query": "\\ub7ad\\uc2a4\\ubbf8\\uc2a4\\uc5d0 \\uad00\\ud574 \\uc54c\\ub824\\uc8fc\\uc138\\uc694."}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-a17cc7a9-b2e3-4d90-8f04-8cb8786baa4a-0', tool_calls=[{'name': 'langchain_search', 'args': {'query': '랭스미스에 관해 알려주세요.'}, 'id': '7ffc7f64-2264-46f3-a2cb-882b01dafa1b', 'type': 'tool_call'}], usage_metadata={'input_tokens': 157, 'output_tokens': 27, 'total_tokens': 1

### 대화 이력을 포함한 에이전트 구현

In [ ]:
!pip install langgraph-checkpoint-sqlite

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string(":memory:") as memory:

  # 대화 이력을 포함한 에이전트 준비
  agent_executor = chat_agent_executor.create_tool_calling_executor(
      llm,
      tools=[tavily_tool, retriever_tool],
      checkpointer=memory
  )

  # 설정 준비
  config = {"configurable": {"thread_id": "abc123"}}

  # 대화
  for chunk in agent_executor.stream(
      {"messages": [("human", "안녕하세요! 제 이름은 승민입니다.")]},
      config=config
  ):
      print(chunk)
      print("--")

  # 설정 준비
  config = {"configurable": {"thread_id": "abc123"}}

  # 과거 대화 내용에 관해 질문
  for chunk in agent_executor.stream(
      {"messages": [("human", "내 이름은?")]},
      config=config
  ):
      print(chunk)
      print("--")

{'agent': {'messages': [AIMessage(content='안녕하세요, 승민님! 만나서 반가워요. 😊 무엇을 도와드릴까요? \n', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-c6a1cad1-fecc-45c0-aeba-22cc71e258c8-0', usage_metadata={'input_tokens': 156, 'output_tokens': 27, 'total_tokens': 183, 'input_token_details': {'cache_read': 0}})]}}
--
{'agent': {'messages': [AIMessage(content='승민님이죠! 😊  제가 기억하고 있어요. \n', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{

In [ ]:
with SqliteSaver.from_conn_string(":memory:") as memory:

  # 대화 이력을 포함한 에이전트 준비
  agent_executor = chat_agent_executor.create_tool_calling_executor(
      llm,
      tools=[tavily_tool, retriever_tool],
      checkpointer=memory
  )

  # 설정 준비
  config = {"configurable": {"thread_id": "abc123"}}

  # 검색 도구를 사용한 질의 응답
  for chunk in agent_executor.stream(
      {"messages": [("human", "랭스미스에 관해 알려주세요.")]},
      config=config
  ):
      print(chunk)
      print("--")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'langchain_search', 'arguments': '{"query": "\\ub7ad\\uc2a4\\ubbf8\\uc2a4\\uc5d0 \\uad00\\ud574 \\uc54c\\ub824\\uc8fc\\uc138\\uc694."}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-de59e82b-5641-44d8-adc2-cf764d95f568-0', tool_calls=[{'name': 'langchain_search', 'args': {'query': '랭스미스에 관해 알려주세요.'}, 'id': 'b040757f-a63a-457b-a8b7-82ff3f7098c1', 'type': 'tool_call'}], usage_metadata={'input_tokens': 157, 'output_tokens': 27, 'total_tokens': 1